In [1]:
from IPython.display import Image

- https://datahonor.com/blog/2025/06/03/llm_kv_cache/

In [3]:
Image(url='https://substackcdn.com/image/fetch/$s_!uVhV!,w_1456,c_limit,f_webp,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F647caf83-cd3d-46f8-8bd0-0946bd896ea1_1023x474.png', 
      width=500)

- MHA: where each head also has its own set of keys and values,
- GQA: to reduce memory usage, GQA groups multiple heads to share the same key and value projections.
- MLA: MLA compresses the key and value tensors into a lower-dimensional space before storing them in the KV cache.
    - At inference time, these compressed tensors are projected back to their original size before being used,
    - This adds an extra matrix multiplication but reduces memory usage.

In [4]:
Image(url='https://substackcdn.com/image/fetch/$s_!jagJ!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2Feb9a75be-2848-4b99-af3d-4c48bdd0181a_1550x858.png'
      , width=500)

### kv cache

- https://huggingface.co/blog/not-lain/kv-caching
    - https://chatgpt.com/c/68c6d894-af24-832f-936e-47eddff37818

In [5]:
# Pseudocode for KV Caching in PyTorch
class KVCache:
    def __init__(self):
        self.cache = {"key": None, "value": None}

    def update(self, key, value):
        if self.cache["key"] is None:
            self.cache["key"] = key
            self.cache["value"] = value
        else:
            self.cache["key"] = torch.cat([self.cache["key"], key], dim=1)
            self.cache["value"] = torch.cat([self.cache["value"], value], dim=1)

    def get_cache(self):
        return self.cache

In [6]:
import torch
torch.manual_seed(0)

def causal_mask(n):
    m = torch.triu(torch.ones(n, n), diagonal=1)
    m = m.masked_fill(m.bool(), float("-inf"))
    return m

def attn(Q, K, V, mask):
    d = Q.size(-1)
    logits = (Q @ K.transpose(-1, -2)) / (d ** 0.5)
    logits = logits + mask
    P = torch.softmax(logits, dim=-1)
    return P @ V, P
    
class SingleHeadSelfAttn(torch.nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.Wq = torch.nn.Linear(d_model, d_model, bias=False)
        self.Wk = torch.nn.Linear(d_model, d_model, bias=False)
        self.Wv = torch.nn.Linear(d_model, d_model, bias=False)
        self.Wo = torch.nn.Linear(d_model, d_model, bias=False)
    def forward(self, X, mask):
        Q = self.Wq(X); K = self.Wk(X); V = self.Wv(X)
        Y, P = attn(Q, K, V, mask)
        return self.Wo(Y), (Q, K, V, P)

In [7]:
d_model = 8
n = 5          # old prefix length
X1 = torch.randn(n, d_model)      # prefix
x2 = torch.randn(1, d_model)      # new single token

layer = SingleHeadSelfAttn(d_model)

In [8]:
# ---- One-step append ----
X_full = torch.cat([X1, x2], dim=0)
mask_full = causal_mask(n+1)
Y_full, (Q_full, K_full, V_full, P_full) = layer(X_full, mask_full)

In [9]:
mask_n = causal_mask(n)
Y_old, (Q1, K1, V1, P1) = layer(X1, mask_n)

In [10]:
q2 = layer.Wq(x2); k2 = layer.Wk(x2); v2 = layer.Wv(x2)
K_cat = torch.cat([K1, k2], dim=0)
V_cat = torch.cat([V1, v2], dim=0)
d = q2.size(-1)
logits_last = (q2 @ K_cat.T) / (d ** 0.5)
P_last = torch.softmax(logits_last, dim=-1)
y_last = layer.Wo(P_last @ V_cat)

In [11]:
torch.allclose(Y_full[:n], Y_old, atol=1e-6, rtol=1e-6)

True

In [12]:
torch.allclose(Y_full[n:n+1], y_last, atol=1e-6, rtol=1e-6)

True

In [13]:
Y_full

tensor([[-0.3981, -0.0212, -0.4192, -0.2045,  0.0336, -0.2520,  0.1180, -0.2676],
        [-0.3965, -0.0201, -0.4471, -0.2247,  0.1494, -0.1615,  0.0856, -0.4335],
        [-0.2937,  0.0643, -0.3283, -0.1094,  0.0022, -0.0951,  0.0720, -0.3483],
        [-0.2506,  0.0644, -0.2712, -0.0973,  0.0297, -0.0602,  0.0370, -0.3269],
        [-0.0726,  0.0102, -0.0993,  0.0124, -0.0306, -0.0194,  0.0489, -0.1278],
        [-0.0991,  0.0736, -0.2615, -0.0349, -0.0542, -0.0130,  0.0555, -0.2092]],
       grad_fn=<MmBackward0>)

In [16]:
Y_old, y_last

(tensor([[-0.3981, -0.0212, -0.4192, -0.2045,  0.0336, -0.2520,  0.1180, -0.2676],
         [-0.3965, -0.0201, -0.4471, -0.2247,  0.1494, -0.1615,  0.0856, -0.4335],
         [-0.2937,  0.0643, -0.3283, -0.1094,  0.0022, -0.0951,  0.0720, -0.3483],
         [-0.2506,  0.0644, -0.2712, -0.0973,  0.0297, -0.0602,  0.0370, -0.3269],
         [-0.0726,  0.0102, -0.0993,  0.0124, -0.0306, -0.0194,  0.0489, -0.1278]],
        grad_fn=<MmBackward0>),
 tensor([[-0.0991,  0.0736, -0.2615, -0.0349, -0.0542, -0.0130,  0.0555, -0.2092]],
        grad_fn=<MmBackward0>))

### mha => gqa

- 几个“查询专家”（Q 头）共享同一份参考资料（K 和 V 头）
    - num_attention_heads：这还是指“查询专家” Q 的总数量。
    - num_kv_heads：这是指 K 和 V 的头的数量。
    - 分组：将 num_attention_heads 个 Q 头分成 num_kv_heads 个组。每个组内的所有 Q 头将共享同一套 K 和 V。
    - 投影：模型仍然为每个 Q 头生成一个独立的 Q 向量。但是，模型只生成 num_kv_heads 组 K 和 V 向量。
- $X \in \mathbb{R}^{L \times d_{model}}$, $N_q$（查询头的数量 (num_attention_heads)）, $N_{kv}$: 键/值头的数量 (num_kv_heads)
    - $g = N_q / N_{kv}$: 每个 KV 头被共享的 Q 头的数量（分组大小）
    - $d_h$: 每个查询头的维度, $d_k$: 每个键/值头的维度 (通常 $d_h = d_k$)，$d_h=\frac{d_{model}}{N_q}$
    - $W_i^Q \in \mathbb{R}^{d_{model} \times d_h}$: 第 $i$ 个查询头的投影矩阵, $i \in \{1, \dots, N_q\}$
    - $W_j^K \in \mathbb{R}^{d_{model} \times d_k}$: 第 $j$ 个键头的投影矩阵, $j \in \{1, \dots, N_{kv}\}$
    - $W_j^V \in \mathbb{R}^{d_{model} \times d_k}$: 第 $j$ 个值头的投影矩阵, $j \in \{1, \dots, N_{kv}\}$
    - $W^O \in \mathbb{R}^{N_q d_h \times d_{model}}$: 输出投影矩阵

#### 计算流程

- 投影

$$
\begin{aligned}
\text{Queries: } & Q_i = X W_i^Q, & \quad \text{for } i = 1, \dots, N_q \\
\text{Keys: } & K_j = X W_j^K, & \quad \text{for } j = 1, \dots, N_{kv} \\
\text{Values: } & V_j = X W_j^V, & \quad \text{for } j = 1, \dots, N_{kv}
\end{aligned}
$$

- 分组与注意力计算 (Grouping and Attention Calculation)
    - 对于第 $i$ 个查询头 $Q_i$, 它需要与一个特定的键/值头对 $(K_j, V_j)$, 进行交互。这个对应关系由分组决定。第 $i$ 个查询头所属的组索引为 $j = \lfloor (i-1) / g \rfloor + 1$
        - 第 $i$ 个头的输出 $\text{Head}_i$ 为：

$$
\text{Head}_i = \text{Attention}(Q_i, K_j, V_j) = \text{softmax}\left(\frac{Q_i K_j^T}{\sqrt{d_k}}\right) V_j
$$

- 拼接与输出 (Concatenation and Final Projection)
    - 所有 $N_q$ 个头的输出被拼接在一起，然后通过最终的输出投影矩阵 $W^O$ 得到最终的输出。

    $$
    \begin{aligned}
    \text{MultiHeadOutput} &= \text{Concat}(\text{Head}_1, \text{Head}_2, \dots, \text{Head}_{N_q})\\
    \text{Output} &= \text{MultiHeadOutput} \cdot W^O
    \end{aligned}
    $$
    - $\text{Concat}(\cdot)$ 操作将所有 $\text{Head}_i \in \mathbb{R}^{L \times d_h}$ 拼接成一个大的矩阵 $\text{MultiHeadOutput} \in \mathbb{R}^{L \times (N_q d_h)}$

*   MHA (Multi-Head Attention): 当 N_kv = N_q 时，分组大小 g=1。每个查询头 Q_i 都对应唯一的键/值对 (K_i, V_i)。这退化为标准的多头注意力机制。
*   MQA (Multi-Query Attention): 当 N_kv = 1 时，分组大小 g=N_q。所有 N_q 个查询头 Q_i 都共享同一对键/值 (K_1, V_1)。这是最高效的 KV 缓存共享模式。
*   GQA (Grouped-Query Attention): 当 1 < N_kv < N_q 时，就是分组查询注意力。它在模型性能和推理效率之间提供了一个灵活的权衡。